In [1]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch.nn.functional import cosine_similarity
import warnings

# Suppress warnings from the transformers library for a cleaner output
warnings.filterwarnings("ignore", category=UserWarning)

def get_contextual_embedding(text: str, target_word: str, model, tokenizer):
    """
    Finds a target word in a text, tokenizes it, and returns the
    contextualized embedding for that specific word from a Transformer model.
    """
    # 1. Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt')

    # 2. Get the model's output (the last hidden state)
    with torch.no_grad():
        outputs = model(**inputs)
        # The shape is [batch_size, num_tokens, embedding_dim]
        last_hidden_state = outputs.last_hidden_state.squeeze(0)

    # 3. Find the token(s) corresponding to our target word
    # This can be tricky if the tokenizer splits the word (e.g., "running" -> "run", "##ing")
    # For this demonstration, we'll use a simple approach for single-token words.
    token_strings = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    target_token_indices = [i for i, token in enumerate(token_strings) if token == target_word]

    if not target_token_indices:
        print(f"Warning: Target word '{target_word}' not found as a single token in '{text}'.")
        return None

    # For simplicity, we take the first occurrence if there are multiple.
    token_index = target_token_indices[0]

    # 4. Extract the embedding vector for the target token
    word_embedding = last_hidden_state[token_index]

    print(f"Text: '{text}'")
    print(f"Found '{target_word}' at token index: {token_index}")

    return word_embedding


In [6]:

def run_disambiguation_test():
    """
    Demonstrates that a Transformer model produces different embeddings for
    the same word used in different contexts.
    """
    print("--- Loading Pre-trained Model (DistilBERT) ---")
    # Use a smaller, efficient model for this demonstration
    model_name = 'distilbert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    print("Model loaded successfully.\n")

    # --- Define Sentences with an Ambiguous Word ---
    # "bank" has a financial meaning here
    sentence_1 = "The manager decided the engineering process for producing thee car."
    # "bank" has a geographical meaning here
    sentence_2 = "The engineering process transforms challenges into solutions by blending creativity, analysis, and precision"

    target_word = "process"

    print("--- Extracting Contextualized Embeddings ---")
    # Get the specific embedding for "bank" from the first sentence
    embedding_1 = get_contextual_embedding(sentence_1, target_word, model, tokenizer)
    print("-" * 20)

    # Get the specific embedding for "bank" from the second sentence
    embedding_2 = get_contextual_embedding(sentence_2, target_word, model, tokenizer)
    print("-" * 20)

    if embedding_1 is None or embedding_2 is None:
        print("\nCould not perform comparison.")
        return

    # --- Compare the Embeddings ---
    # The two embedding vectors must be reshaped to (1, D) for cosine_similarity
    similarity = cosine_similarity(embedding_1.unsqueeze(0), embedding_2.unsqueeze(0)).item()

    print("\n--- Analysis ---")
    print(f"Cosine Similarity between the two vectors for the word 'bank': {similarity:.4f}")

    if similarity < 0.7:
        print("✅ Result: The similarity is LOW, as expected.")
        print("This demonstrates that the model generated two distinct, context-aware embeddings for the same word.")
    else:
        print("❌ Result: The similarity is HIGH.")
        print("This would indicate the model did not effectively disambiguate the word based on context.")


In [7]:

if __name__ == '__main__':
    run_disambiguation_test()

--- Loading Pre-trained Model (DistilBERT) ---
Model loaded successfully.

--- Extracting Contextualized Embeddings ---
Text: 'The manager decided the engineering process for producing thee car.'
Found 'process' at token index: 6
--------------------
Text: 'The engineering process transforms challenges into solutions by blending creativity, analysis, and precision'
Found 'process' at token index: 3
--------------------

--- Analysis ---
Cosine Similarity between the two vectors for the word 'bank': 0.7331
❌ Result: The similarity is HIGH.
This would indicate the model did not effectively disambiguate the word based on context.
